<div dir="rtl">

# שלב 3: אימון Reranker לעברית

**מה זה Reranker:** מודל שמקבל את השאלה **ואת הפסקה יחד**, וקורא אותן כטקסט אחד. בתוך המודל, כל מילה בשאלה "רואה" כל מילה בפסקה, ובסוף יוצא ציון אחד: כמה הפסקה עונה על השאלה. זה מדויק יותר ממודל וקטורים, שדוחס כל טקסט לווקטור בנפרד, אבל גם איטי יותר, כי אי אפשר לחשב שום דבר מראש.

**לכן החיפוש נעשה בשני שלבים:**
1. החיפוש ההיברידי (BM25 + וקטורים) מביא את 20 המועמדות הטובות ביותר מתוך כל המאגר.
2. ה-Reranker קורא רק את ה-20 האלה, ומסדר אותן מחדש.

**מה המחברת עושה:**
1. מכינה נתוני אימון: לכל שאלה, הפסקה הנכונה ו-3 פסקאות שליליות קשות מתוך תוצאות החיפוש ההיברידי.
2. מאמנת את `BAAI/bge-reranker-v2-m3`, Reranker רב-לשוני, **על שני כרטיסי ה-GPU במקביל**.
3. מודדת את ה-Reranker לפני ואחרי האימון, על גבי המערכת הסופית שלנו: חיפוש היברידי עם המודל המאומן בלי הסינון.
4. מעלה את המודל ואת התוצאות ל-Hugging Face.

**לפני ההרצה ב-Kaggle:**
- `Settings → Accelerator → GPU T4 x2`, ו-`Internet` מופעל.
- `Add-ons → Secrets`: סוד בשם `HF_TOKEN` עם טוקן בהרשאת Write, מסומן כזמין למחברת הזאת.
- הרצה ברקע: `Save Version → Save & Run All (Commit)`. אפשר לסגור את הדפדפן. בערך שעה עד שעה וחצי.

</div>

<div dir="rtl">

## 1. התקנה, הגדרות והתחברות

</div>

In [ ]:
!pip install -q -U sentence-transformers rank_bm25

In [ ]:
import os, json, random, re, time, gc
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda"
N_GPUS = torch.cuda.device_count()
print(f"GPUs: {N_GPUS} x {torch.cuda.get_device_name(0)}")

RERANKER_BASE = "BAAI/bge-reranker-v2-m3"
RERANKER_NAME = "hebrew-qa-reranker"
RETRIEVER = "intfloat/multilingual-e5-large"
FT_RETRIEVER = "hebrew-qa-retriever-e5-large-nofilter"   # our final retriever (private models work, we are logged in)
DATASET_NAME = "hebrew-qa-retrieval-data"
N_NEG, TOP_K, MAX_LEN = 3, 20, 384
EPOCHS, BATCH_PER_GPU, LR = 1, 16, 1e-5

In [ ]:
from huggingface_hub import login, HfApi
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    raise RuntimeError("Add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets) and attach it to this notebook") from e
api = HfApi()
user = api.whoami()["name"]
data_repo = f"{user}/{DATASET_NAME}"
print("Logged in as", user)

<div dir="rtl">

## 2. טעינת HeQ

</div>

In [ ]:
!git clone -q --depth 1 https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset heq 2>/dev/null || echo "already cloned"

corpus, para_id = [], {}
queries = {"train": [], "val": [], "test": []}
for split, fname in [("train", "train v1.1.json"), ("val", "val v1.1.json"), ("test", "test v1.1.json")]:
    data = json.load(open(f"heq/data/data v1.1/{fname}", encoding="utf-8"))["data"]
    for article in data:
        for p in article["paragraphs"]:
            text = p["context"].strip()
            if text not in para_id:
                para_id[text] = len(corpus)
                corpus.append({"id": len(corpus), "title": article["title"],
                               "source": article["source"], "split": split, "text": text})
            pid = para_id[text]
            for q in p["qas"]:
                if not q["is_impossible"]:
                    queries[split].append({"qid": q["id"], "question": q["question"].strip(),
                                           "pid": pid, "source": article["source"]})
corpus_texts = [p["text"] for p in corpus]
test_q = queries["test"]
print(f"Paragraphs: {len(corpus):,} | " + " | ".join(f"{s}: {len(v):,}" for s, v in queries.items()))

<div dir="rtl">

## 3. שלב ראשון: חיפוש היברידי

המערכת הסופית שלנו: BM25 והמודל המאומן בלי הסינון, משולבים ב-Reciprocal Rank Fusion. מחשבים אותה:
- **לשאלות האימון:** כדי לבחור שליליים קשים. כך ה-Reranker מתאמן בדיוק על סוג המועמדות שהוא יקבל בפועל. החיפוש נעשה רק בתוך פסקאות האימון.
- **לשאלות הבדיקה:** כדי לקבל את 20 המועמדות שה-Reranker יסדר.

בנוסף מחשבים את החיפוש ההיברידי עם e5-large **בלי אימון**, כדי לראות כמה ה-Reranker מוסיף גם למערכת בלי מודל מאומן.

</div>

In [ ]:
from rank_bm25 import BM25Okapi
tokenize = lambda t: re.findall(r"\w+", t.lower())

def to_ranks(S):
    # rank (1 = best) of every paragraph for every query
    order = S.argsort(dim=1, descending=True)
    r = torch.empty_like(order)
    r.scatter_(1, order, torch.arange(1, S.shape[1] + 1, device=S.device).expand_as(order))
    return r.float()

def bm25_matrix(bm25, qs):
    return torch.tensor(np.array([bm25.get_scores(tokenize(q["question"])) for q in tqdm(qs, desc="BM25")]),
                        device=device, dtype=torch.float32)

def rrf(B, Q, C, k=60):
    return 1 / (k + to_ranks(B)) + 1 / (k + to_ranks(Q @ C.T))

def encode(model, texts, prompt, bs=64):
    return model.encode(texts, batch_size=bs, convert_to_tensor=True, normalize_embeddings=True,
                        show_progress_bar=True, prompt=prompt).float()

def embed(name):
    m = SentenceTransformer(name, device=device).half()
    m.max_seq_length = 512
    out = (encode(m, corpus_texts, "passage: "),
           encode(m, [q["question"] for q in queries["train"]], "query: "),
           encode(m, [q["question"] for q in test_q], "query: "))
    del m; gc.collect(); torch.cuda.empty_cache()
    return out

FT_LABEL, BASE_LABEL = "fine-tuned e5-large", "e5-large, no training"
emb = {BASE_LABEL: embed(RETRIEVER)}
try:
    emb[FT_LABEL] = embed(f"{user}/{FT_RETRIEVER}")
except Exception as e:
    print(f"Could not load {user}/{FT_RETRIEVER} ({type(e).__name__}); continuing with the untrained model only")
MINER = FT_LABEL if FT_LABEL in emb else BASE_LABEL
print("Hard negatives are mined with:", MINER)

# Test: hybrid over the full corpus, for each retriever
bm25_all = BM25Okapi([tokenize(t) for t in corpus_texts])
B_test = bm25_matrix(bm25_all, test_q)
H_test = {name: rrf(B_test, Qte, C) for name, (C, Qtr, Qte) in emb.items()}

# Train: hybrid over training paragraphs only, with the mining retriever
train_pids = [p["id"] for p in corpus if p["split"] == "train"]
bm25_train = BM25Okapi([tokenize(corpus_texts[i]) for i in train_pids])
C, Qtr, _ = emb[MINER]
H_train = rrf(bm25_matrix(bm25_train, queries["train"]), Qtr, C[train_pids])
top_train = H_train.topk(30, dim=1).indices.cpu()
del H_train; torch.cuda.empty_cache()

<div dir="rtl">

## 4. נתוני אימון

לכל שאלה: זוג אחד של (שאלה, פסקה נכונה) עם תווית 1, ו-3 זוגות של (שאלה, פסקה שלילית) עם תווית 0. את השליליות בוחרים באקראי מתוך 20 התוצאות הראשונות של החיפוש ההיברידי, כלומר בדיוק מסוג הפסקאות שה-Reranker יצטרך להבחין ביניהן. **בלי סינון של פסקאות מאותו ערך**, כי ניסוי ההשוואה הראה שבמודל הווקטורים הן היו השליליים המועילים ביותר.

</div>

In [ ]:
pairs = []
for i, q in enumerate(tqdm(queries["train"], desc="Building pairs")):
    cands = [train_pids[j] for j in top_train[i].tolist() if train_pids[j] != q["pid"]][:TOP_K]
    pairs.append({"query": q["question"], "passage": corpus[q["pid"]]["text"], "label": 1.0})
    for pid in random.sample(cands, min(N_NEG, len(cands))):
        pairs.append({"query": q["question"], "passage": corpus[pid]["text"], "label": 0.0})
random.shuffle(pairs)
with open("train_pairs.jsonl", "w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Training pairs: {len(pairs):,} ({sum(p['label'] for p in pairs):,.0f} positive)")

<div dir="rtl">

## 5. אימון על שני כרטיסים

כאן מנצלים את שני כרטיסי ה-T4. את קוד האימון כותבים לקובץ, ומריצים אותו עם `torchrun`, שמפעיל עותק אחד על כל כרטיס. כל עותק מקבל חצי מהנתונים, ובסוף כל צעד הם מסנכרנים ביניהם את העדכונים, כך שבפועל זה אימון אחד, כמעט פי שניים מהר יותר.

ב-Reranker זה לא משנה את האימון עצמו, כי כל זוג נבדק לבד, בלי "שליליים מתוך ה-batch". לכן שני כרטיסים כאן הם רק האצה.

- **Loss:** `BinaryCrossEntropyLoss`: המודל לומד לתת ציון גבוה לזוג נכון ונמוך לזוג שגוי. יש 3 שליליים על כל חיובי, אז נותנים לחיובי משקל פי 3, כדי שהמודל לא ילמד פשוט לענות "לא" תמיד.
- **Gradient checkpointing:** חוסך זיכרון בכרטיס, במחיר של קצת זמן.

</div>

In [ ]:
%%writefile train_reranker.py
import sys, torch
from datasets import load_dataset
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

base, out_dir, epochs, batch, lr, max_len, n_neg = sys.argv[1:8]
model = CrossEncoder(base, num_labels=1, max_length=int(max_len))

def enable_gradient_checkpointing(m):
    # find the underlying Hugging Face model and turn on gradient checkpointing (saves GPU memory)
    for mod in m.modules():
        if mod is not m and hasattr(mod, "gradient_checkpointing_enable"):
            mod.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            return True
    return False
print("Gradient checkpointing:", enable_gradient_checkpointing(model))
train = load_dataset("json", data_files="train_pairs.jsonl", split="train")
loss = BinaryCrossEntropyLoss(model, pos_weight=torch.tensor(float(n_neg)))
args = CrossEncoderTrainingArguments(
    output_dir="checkpoints-reranker", num_train_epochs=float(epochs), per_device_train_batch_size=int(batch),
    learning_rate=float(lr), warmup_ratio=0.1, fp16=True,
    save_strategy="no", logging_steps=50, report_to="none", seed=42,
    dataloader_drop_last=True, ddp_find_unused_parameters=False)
trainer = CrossEncoderTrainer(model=model, args=args, train_dataset=train, loss=loss)
trainer.train()
if trainer.is_world_process_zero():
    model.save(out_dir)
    print("Saved to", out_dir)

In [ ]:
gc.collect(); torch.cuda.empty_cache()
start = time.time()
!torchrun --nproc_per_node={N_GPUS} train_reranker.py {RERANKER_BASE} reranker-ft {EPOCHS} {BATCH_PER_GPU} {LR} {MAX_LEN} {N_NEG}
print(f"Training took {(time.time() - start) / 60:.1f} minutes")
assert os.path.exists("reranker-ft"), "Training failed: see the log above"

In [ ]:
# Back up the trained reranker to Hugging Face right away
from sentence_transformers.cross_encoder import CrossEncoder
CrossEncoder("reranker-ft").push_to_hub(f"{user}/{RERANKER_NAME}", exist_ok=True)
print("Backed up:", f"https://huggingface.co/{user}/{RERANKER_NAME}")

<div dir="rtl">

## 6. מדידה

לכל שאלת בדיקה לוקחים את 20 המועמדות מהחיפוש ההיברידי, וה-Reranker מסדר אותן מחדש. אם הפסקה הנכונה לא הייתה ב-20 הראשונות, ה-Reranker לא יכול לעזור, והמיקום שלה נשאר כמו שהיה. לכן השורה **"Recall@20 של השלב הראשון"** היא התקרה: הציון הכי גבוה שה-Reranker יכול להגיע אליו.

מודדים:
- חיפוש היברידי בלבד, בלי Reranker.
- היברידי + ה-Reranker המקורי, בלי אימון.
- היברידי + ה-Reranker שלנו, אחרי אימון.

את שלוש השורות מודדים פעמיים: על המערכת הסופית (עם המודל המאומן), ועל החיפוש ההיברידי עם e5-large בלי אימון.

</div>

In [ ]:
def metrics_from_ranks(r):
    r = np.asarray(r)
    return {"R@1": np.mean(r <= 1) * 100, "R@5": np.mean(r <= 5) * 100, "R@10": np.mean(r <= 10) * 100,
            "MRR@10": np.mean(np.where(r <= 10, 1 / r, 0)) * 100,
            "NDCG@10": np.mean(np.where(r <= 10, 1 / np.log2(r + 1), 0)) * 100}

pos = torch.tensor([q["pid"] for q in test_q], device=device)

def first_stage(H):
    ranks = ((H > H[torch.arange(len(pos), device=device), pos][:, None]).sum(1) + 1).cpu().numpy()
    cands = H.topk(TOP_K, dim=1).indices.cpu().numpy()
    return ranks, cands

def rerank(reranker, ranks, cands):
    pairs = [(q["question"], corpus_texts[c]) for q, row in zip(test_q, cands) for c in row]
    scores = np.asarray(reranker.predict(pairs, batch_size=64, show_progress_bar=True)).reshape(len(test_q), TOP_K)
    new = ranks.copy()
    for i, q in enumerate(test_q):
        hit = np.where(cands[i] == q["pid"])[0]
        if len(hit):
            new[i] = 1 + (scores[i] > scores[i, hit[0]]).sum()
    return new

rows = []
def add(name, ranks):
    rows.append({"Model": name, **metrics_from_ranks(ranks)})
    display(pd.DataFrame(rows).set_index("Model").round(1))

stages = {f"Hybrid (BM25 + {name})": H for name, H in H_test.items()}

rerankers = {"bge-reranker-v2-m3 (no training)": CrossEncoder(RERANKER_BASE, max_length=512, device=device),
             "fine-tuned reranker (ours)": CrossEncoder("reranker-ft", max_length=512, device=device)}
for r in rerankers.values():
    r.half()

for stage_name, H in stages.items():
    ranks, cands = first_stage(H)
    print(f"{stage_name}: correct paragraph in the top {TOP_K} for {np.mean(ranks <= TOP_K) * 100:.1f}% of questions")
    add(stage_name, ranks)
    for r_name, r in rerankers.items():
        add(f"{stage_name} + {r_name}", rerank(r, ranks, cands))

results = pd.DataFrame(rows).set_index("Model").round(1)
results.to_csv("results_reranker.csv")
results

<div dir="rtl">

## 7. העלאה ל-Hugging Face

</div>

In [ ]:
repo = f"{user}/{RERANKER_NAME}"
table = results.reset_index().to_markdown(index=False)
card = f'''---
language: [he]
license: apache-2.0
library_name: sentence-transformers
base_model: {RERANKER_BASE}
datasets: [HeQ]
tags: [sentence-transformers, cross-encoder, reranker, hebrew, rag]
pipeline_tag: text-ranking
---

# Hebrew QA Reranker (fine-tuned bge-reranker-v2-m3)

A cross-encoder that scores how well a Hebrew paragraph answers a Hebrew question. Fine-tuned from
`{RERANKER_BASE}` on the answerable questions of [HeQ](https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset),
with {N_NEG} hard negatives per question taken from a hybrid BM25 + {MINER} first stage, and
`BinaryCrossEntropyLoss` ({EPOCHS} epoch, lr {LR}, 2 x T4).

Use it as the second stage of a search: retrieve the top {TOP_K} paragraphs, then rerank them.

```python
from sentence_transformers.cross_encoder import CrossEncoder
model = CrossEncoder("{repo}")
scores = model.predict([("מי ניסח את מגילת העצמאות?", "paragraph 1"), ("מי ניסח את מגילת העצמאות?", "paragraph 2")])
```

## Results (HeQ test set, {len(test_q):,} questions, reranking the top {TOP_K} of {len(corpus):,} paragraphs)

{table}

Test articles are disjoint from training articles. HeQ is licensed CC BY 4.0 (Webiks / MAFAT).
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
api.upload_file(path_or_fileobj="results_reranker.csv", path_in_repo="results_reranker.csv",
                repo_id=data_repo, repo_type="dataset")
print("Model:  ", f"https://huggingface.co/{repo}")
print("Results:", f"https://huggingface.co/datasets/{data_repo}")